# StoneSense-AI: CT Multi-Model Experimentation — YOLO26 Classification

### 📋 Overview
This notebook trains and evaluates the **Ultralytics YOLO26** classification model (`yolo26s-cls.pt` or `yolo26n-cls.pt`) on `dl/datasets/yolo_view`.
* Note on License: Ultralytics models are released under **AGPL-3.0**.
* Uses `yolo26s-cls.pt` backbone (30 epochs, or 1 epoch in `SMOKE_TEST` mode, imgsz 224).
* Generates batch predictions on `validation` and `test` splits and computes metrics strictly via `common_eval.py` (not internal Ultralytics metrics) for direct comparability with ResNet18 and DINOv3.
* Exports contract artifacts: `model_card.json`, `metrics.json`, `predictions.csv`, `best.pt`, `predict_example.py`, and `yolo26_artifacts.zip`.

---
### 1. Global Configuration & Smoke Test Settings
* **What it does:** Configures hyperparameters, epochs, batch size, paths, seed, and `SMOKE_TEST` flag (1 epoch, 50 samples).
* **Expected runtime:** < 1 second.
* **Expected output:** Configuration summary.

In [ ]:
import os
import sys
import time
from pathlib import Path

SEED = 42
RERUN = False
SMOKE_TEST = False  # Set to True for fast 1-epoch / 50-sample verification
EPOCHS = 1 if SMOKE_TEST else 30
BATCH_SIZE = 16 if SMOKE_TEST else 32
BOOTSTRAP_RESAMPLES = 20 if SMOKE_TEST else 1000
IMGSZ = 224
FAMILY = "yolo26"
VERSION_TAG = "yolo26s_v001"
BASE_WEIGHT = "yolo26s-cls.pt"

WORKSPACE_ROOT = Path("/content/StoneSense-AI") if "google.colab" in sys.modules else Path(".").resolve()
sys.path.insert(0, (WORKSPACE_ROOT / "dl" / "notebooks").as_posix())

YOLO_DATA_YAML = WORKSPACE_ROOT / "dl" / "datasets" / "yolo_view" / "dataset.yaml"
MANIFEST_PATH = WORKSPACE_ROOT / "dl" / "outputs" / "reports" / "grouped_split_manifest.csv"
OUT_DIR = WORKSPACE_ROOT / "dl" / "models" / "ct"
RUNS_DIR = WORKSPACE_ROOT / "dl" / "artifacts" / "yolo_runs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace Root: {WORKSPACE_ROOT.as_posix()}")
print(f"YOLO Dataset Config: {YOLO_DATA_YAML.as_posix()}")
print(f"Output Directory: {(OUT_DIR / FAMILY).as_posix()}")
print(f"Smoke Test Mode: {SMOKE_TEST} (Epochs: {EPOCHS})")

---
### 2. Check Cell: Environment, GPU & Ultralytics Weights Verification
* **What it does:** Verifies `ultralytics` installation, GPU acceleration, and tests loading base weights `yolo26s-cls.pt` (with fallback to `yolo11s-cls.pt` if needed).
* **Expected runtime:** ~5-10 seconds.
* **Expected output:** Loaded YOLO model confirmation and AGPL-3.0 license declaration.

In [ ]:
import torch
from ultralytics import YOLO

device = "0" if torch.cuda.is_available() else "cpu"
print(f"Execution Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

try:
    print(f"Testing base weight loading: {BASE_WEIGHT}...")
    test_model = YOLO(BASE_WEIGHT)
    active_weight = BASE_WEIGHT
    display_name = "Ultralytics YOLO26-Small Classifier"
    print(f"Successfully initialized YOLO26: {BASE_WEIGHT}")
except Exception as e:
    print(f"YOLO26 weights not found ({e}); testing YOLO11 fallback...")
    active_weight = "yolo11s-cls.pt"
    test_model = YOLO(active_weight)
    display_name = "Ultralytics YOLO11-Small Classifier (Fallback)"
    print(f"Loaded fallback weight: {active_weight}")

print("=" * 60)
print(f"Model Display Name: {display_name}")
print("License Notice: AGPL-3.0 (Ultralytics Open Source License)")
print("=" * 60)

---
### 3. Train YOLO26 Classifier
* **What it does:** Fits YOLO26 classification head on `train` split (30 epochs or 1 epoch in SMOKE_TEST) with validation monitoring.
* **Expected runtime:** ~10-15 mins on T4 GPU (~1 min in SMOKE_TEST). Idempotent: skips if `best.pt` exists.
* **Expected output:** Training epoch progress and saved `best.pt` path.

In [ ]:
family_dir = OUT_DIR / FAMILY
family_dir.mkdir(parents=True, exist_ok=True)
best_pt_path = family_dir / "best.pt"

start_train_time = time.time()

if best_pt_path.exists() and not RERUN and not SMOKE_TEST:
    print(f"Found existing checkpoint at {best_pt_path.as_posix()}. Skipping training.")
    model = YOLO(best_pt_path)
    training_duration = 0.0
else:
    print(f"Launching YOLO training ({EPOCHS} epochs, batch {BATCH_SIZE}, imgsz {IMGSZ})...")
    model = YOLO(active_weight)
    
    results = model.train(
        data=(WORKSPACE_ROOT / "dl" / "datasets" / "yolo_view").as_posix(),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        seed=SEED,
        device=device,
        project=RUNS_DIR.as_posix(),
        name=FAMILY,
        exist_ok=True,
        verbose=True
    )
    training_duration = time.time() - start_train_time
    
    # Copy best.pt to family output directory
    trained_best = RUNS_DIR / FAMILY / "weights" / "best.pt"
    if trained_best.exists():
        import shutil
        shutil.copy(trained_best, best_pt_path)
        print(f"Saved champion weights to {best_pt_path.as_posix()}")
    else:
        raise FileNotFoundError(f"Training output best.pt not found at {trained_best.as_posix()}")

print(f"YOLO training process complete ({training_duration:.1f}s).")

---
### 4. Batch Prediction & Common Evaluation Pipeline
* **What it does:** Runs raw inference on `validation` and `test` images to get class probability vectors `[p_Cyst, p_Normal, p_Stone, p_Tumor]`, evaluates with `common_eval.py` and cluster bootstrap 95% CIs.
* **Expected runtime:** ~1-2 mins (~5s in SMOKE_TEST).
* **Expected output:** Comprehensive performance report and comparison with CIs.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from common_eval import generate_full_evaluation, CLASS_NAMES

df_manifest = pd.read_csv(MANIFEST_PATH)
trained_model = YOLO(best_pt_path)

def predict_split(split_name: str) -> pd.DataFrame:
    sub_df = df_manifest[df_manifest["destination_split"] == split_name].reset_index(drop=True)
    if SMOKE_TEST:
        sub_df = sub_df.head(50)
    all_probs = []
    
    print(f"Running inference on '{split_name}' split ({len(sub_df):,} images)...")
    for _, row in tqdm(sub_df.iterrows(), total=len(sub_df)):
        img_p = WORKSPACE_ROOT / "dl" / "processed_grouped" / split_name / row["class"] / row["filename"]
        
        res = trained_model(img_p.as_posix(), verbose=False)[0]
        probs = res.probs.data.cpu().numpy()
        
        yolo_names = res.names
        prob_dict = {yolo_names[i]: float(probs[i]) for i in range(len(probs))}
        canonical_probs = [prob_dict.get(cname, 0.0) for cname in CLASS_NAMES]
        all_probs.append(canonical_probs)
        
    prob_mat = np.array(all_probs)
    pred_classes = [CLASS_NAMES[idx] for idx in np.argmax(prob_mat, axis=1)]
    
    return pd.DataFrame({
        "filename": sub_df["filename"],
        "cluster_id": sub_df["near_duplicate_group_id"],
        "y_true": sub_df["class"],
        "y_pred": pred_classes,
        "p_Cyst": prob_mat[:, 0],
        "p_Normal": prob_mat[:, 1],
        "p_Stone": prob_mat[:, 2],
        "p_Tumor": prob_mat[:, 3],
    })

df_val_preds = predict_split("validation")
df_test_preds = predict_split("test")

eval_results = generate_full_evaluation(df_val_preds, df_test_preds, bootstrap_iterations=BOOTSTRAP_RESAMPLES, seed=SEED)

print("=" * 60)
print(f"TEST SPLIT EVALUATION (YOLO26):")
print(f"   - Slice Accuracy    : {eval_results['test']['slice_accuracy']:.4f}")
print(f"   - Slice Macro F1    : {eval_results['test']['slice_macro_f1']:.4f} (95% CI: [{eval_results['test']['confidence_intervals_95']['slice_macro_f1']['ci_lower_95']:.4f}, {eval_results['test']['confidence_intervals_95']['slice_macro_f1']['ci_upper_95']:.4f}])")
print(f"   - Cluster Macro F1  : {eval_results['test']['cluster_macro_f1']:.4f}")
print(f"   - Stone Recall      : {eval_results['test']['per_class']['Stone']['recall']:.4f}")
print(f"   - Tumor Recall      : {eval_results['test']['per_class']['Tumor']['recall']:.4f}")
print("=" * 60)

---
### 5. Save Artifacts Contract & `predict_example.py`
* **What it does:** Saves `model_card.json`, `metrics.json`, `predictions.csv`, `best.pt`, and `predict_example.py` under `dl/models/ct/yolo26/`.
* **Expected runtime:** ~2-3 seconds.
* **Expected output:** SHA-256 manifest and artifact summary.

In [ ]:
from common_eval import save_contract_artifacts

predict_example_code = '''"""Standalone YOLO26 prediction example for Phase 8 contract validation."""
from pathlib import Path
from ultralytics import YOLO

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]

def predict_single_image(image_path: str, model_dir: str = ".") -> dict:
    model_path = Path(model_dir) / "best.pt"
    model = YOLO(model_path.as_posix())
    results = model(image_path, verbose=False)[0]
    probs = results.probs.data.cpu().numpy()
    yolo_names = results.names
    prob_dict = {yolo_names[i]: float(probs[i]) for i in range(len(probs))}
    return {c: float(prob_dict.get(c, 0.0)) for c in CLASS_NAMES}

if __name__ == "__main__":
    import sys
    img_file = sys.argv[1] if len(sys.argv) > 1 else "sample.jpg"
    print(f"Probabilities: {predict_single_image(img_file)}")
'''

saved = save_contract_artifacts(
    out_dir=OUT_DIR,
    family=FAMILY,
    version_tag=VERSION_TAG,
    display_name=display_name,
    base_model=active_weight,
    license_name="AGPL-3.0",
    hyperparameters={"epochs": EPOCHS, "batch_size": BATCH_SIZE, "imgsz": IMGSZ, "seed": SEED, "smoke_test": SMOKE_TEST},
    preprocessing_spec={"resize": [IMGSZ, IMGSZ], "normalization": "Ultralytics standard"},
    library_versions={"torch": torch.__version__, "ultralytics": "installed"},
    training_time_sec=training_duration,
    evaluation_results=eval_results,
    additional_artifact_paths=[best_pt_path],
    predict_example_code=predict_example_code,
    seed=SEED
)

print(f"Successfully saved YOLO26 Phase 8 contract artifacts to {family_dir.as_posix()}")

---
### 6. Package Zip for Project Integration
* **What it does:** Zips `dl/models/ct/yolo26/` into `yolo26_artifacts.zip`.
* **Expected runtime:** ~2-5 seconds.
* **Expected output:** Downloadable zip confirmation.

In [ ]:
import shutil

zip_path = WORKSPACE_ROOT / "dl" / "models" / "ct" / f"{FAMILY}_artifacts.zip"
shutil.make_archive(str(zip_path.with_suffix("")), 'zip', family_dir)

print("=" * 60)
print(f"CREATED ARTIFACT ZIP: {zip_path.name}")
print(f"Target Project Unzip Directory: dl/models/ct/{FAMILY}/")
print("=" * 60)